In [1]:
import sys
sys.path.append("../")
from tempfile import TemporaryDirectory
import random
from pathlib import Path
from PIL import Image
import torch
from clearml import Model
from timm import create_model
from timm.data import resolve_model_data_config, create_transform

In [2]:
tempdir = TemporaryDirectory()

In [3]:
tempdir

<TemporaryDirectory '/tmp/tmpi5v18kfj'>

In [4]:
model_id = Model.query_models(project_name="ImageClassifier", model_name="ImageClassifier", tags=["best"], max_results=1)[0].id
model_id

'7fd95e31f3ee434a81fe048bcc9e243d'

In [5]:
clearml_model = Model(model_id)
model_path = Path(clearml_model.get_local_copy())

In [6]:
model_path

PosixPath('/home/ghisso/.clearml/cache/storage_manager/global/2d3de16deb18b94dcf1a4c12be161ed2.model.pth')

In [7]:
metadata = clearml_model.get_all_metadata_casted()
metadata

{'model_name': 'resnet18.a1_in1k', 'num_classes': 9, 'in_chans': 3}

In [8]:
loaded_model = create_model(**metadata, checkpoint_path=model_path)

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [10]:
loaded_model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, m

In [11]:
data_config = resolve_model_data_config(loaded_model)
data_config

{'input_size': (3, 224, 224),
 'interpolation': 'bicubic',
 'mean': (0.485, 0.456, 0.406),
 'std': (0.229, 0.224, 0.225),
 'crop_pct': 0.95,
 'crop_mode': 'center'}

In [12]:
inference_transforms = create_transform(**data_config, is_training=False, separate=False)
if isinstance(inference_transforms, tuple):
    inference_transforms = inference_transforms[0]

In [13]:
inference_transforms

Compose(
    Resize(size=235, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

In [14]:
test_folder = Path("../../data/splits/test")

In [15]:
label2id = clearml_model.labels
id2label = {v:k for k,v in label2id.items()}

In [16]:
label2id

{'apple': 0,
 'banana': 1,
 'cherry': 2,
 'chickoo': 3,
 'grapes': 4,
 'kiwi': 5,
 'mango': 6,
 'orange': 7,
 'strawberry': 8}

In [17]:
id2label

{0: 'apple',
 1: 'banana',
 2: 'cherry',
 3: 'chickoo',
 4: 'grapes',
 5: 'kiwi',
 6: 'mango',
 7: 'orange',
 8: 'strawberry'}

### Inference

In [18]:
def infer_image(image_path: Path) ->int:
    image = Image.open(image_path).convert("RGB")
    input = inference_transforms(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = loaded_model(input)
    return torch.argmax(output).item()

In [19]:
test_images = list(test_folder.glob("*/*"))

In [20]:
len(test_images)

36

In [21]:
idx = random.randint(0, len(test_images))
print(test_images[idx])
print(f"This is an image of {id2label[infer_image(test_images[idx])]}")

../../data/splits/test/kiwi/Image_9.jpg
This is an image of kiwi


In [22]:
def infer_image_softmax(image_path: Path) ->int:
    image = Image.open(image_path).convert("RGB")
    input = inference_transforms(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = loaded_model(input)
    return torch.softmax(output, dim=1).tolist()

In [23]:
infer_image_softmax(test_images[idx])

[[0.11480334401130676,
  0.09167174994945526,
  0.10389754176139832,
  0.11804094165563583,
  0.1021014079451561,
  0.12514638900756836,
  0.10565857589244843,
  0.11969785392284393,
  0.11898218840360641]]

In [24]:
batch_size = 8
sample_paths = random.sample(test_images, k=min(batch_size, len(test_images)))

loaded_model.eval()
batch = torch.stack([inference_transforms(Image.open(p).convert("RGB")) for p in sample_paths]).to(device)

with torch.no_grad():
    logits = loaded_model(batch)
    pred_ids = torch.argmax(logits, dim=1).cpu().tolist()

print("Batch inference results:")
for path, pred_id in zip(sample_paths, pred_ids):
    real_class = path.parent.name
    pred_class = id2label[pred_id]
    print(f"{path.name} | predicted: {pred_class} | real: {real_class}")

Batch inference results:
Image_40.jpg | predicted: cherry | real: chickoo
Image_6.jpg | predicted: orange | real: chickoo
Image_9.jpg | predicted: apple | real: apple
Image_6.jpg | predicted: strawberry | real: strawberry
Image_30.jpg | predicted: apple | real: apple
Image_40.png | predicted: banana | real: banana
Image_6.jpg | predicted: apple | real: cherry
Image_6.jpg | predicted: kiwi | real: kiwi
